<a href="https://colab.research.google.com/github/oisincam/quantum_circuits/blob/main/stim_stabilizer_groups.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Introduction

This notebook provides some useful tools to manipulates stabilizer groups; we can move between representations, compute logical operations, and check commutation properties.

In [ ]:
!pip install stim
!pip install galois

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 41.1 MB/s eta 0:00:00


In [ ]:
import stim
import numpy as np
import galois

##Representation Conversion

We can represent a stabilizer group using string, or as a stim object, or as a binary matrix.

In [ ]:
#convert a space delimited string of length 2n from magma into a stim Pauli string of length n
def SpacedStringToPauli(spaced_string):
  int_list=[int(x) for x in spaced_string.split()] # turn string into list, using space delimiters.
  n=len(int_list)
  if n%2==1:
    raise ValueError('The length of the spaced string must be even')
  xlist=[]
  zlist=[]
  for i in range(0,int(n/2)):
    xlist.append(int_list[i]) #first half of list goes in x part
    zlist.append(int_list[int(n/2)+i])# second part of list goes in z part

  xs = np.array(xlist, dtype=np.bool_)
  zs = np.array(zlist, dtype=np.bool_)
  pauli = stim.PauliString.from_numpy(xs=xs, zs=zs, sign=1) #use x and z part to make stimPauli
  return pauli

In [ ]:
#convert a list of normal strings into stim pauli strings
def StringToStimPauli(list_of_strings):
  list_of_stimPaulis=[]
  for string in list_of_strings:
    list_of_stimPaulis.append(stim.PauliString(string))
  return list_of_stimPaulis


#Example
stabilizer=['XZZXI','IXZZX','XIXZZ','ZXIXZ']
StringToStimPauli(stabilizer)


[stim.PauliString("+XZZX_"),
 stim.PauliString("+_XZZX"),
 stim.PauliString("+X_XZZ"),
 stim.PauliString("+ZX_XZ")]

In [ ]:
#convert a list of stimPaulis to a galois matrix
def StimPaulisToMatrix(list_of_stimPaulis):
  matrix_rows=[]
  for stimPauli in list_of_stimPaulis:
    xpart,zpart=stimPauli.to_numpy() # use stim to convert binary, gives numpy arrays
    xpart=xpart*1
    zpart=zpart*1
    xlist=xpart.tolist()
    zlist=zpart.tolist()
    row=xlist+zlist #conccatenate the x and z part
    matrix_rows.append(row)
  GF2=galois.GF(2)
  matrix=GF2(matrix_rows) #make a matrix from the rows
  return matrix


#Example
StimPaulisToMatrix(StringToStimPauli(['IXXYZZ', 'XXYYII']))

GF([[0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1],
    [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0]], order=2)

In [ ]:
#convert a galois matrix to a list of stimPaulis
def MatrixToStimPaulis(matrix):
  m,two_n=matrix.shape
  n=int(two_n/2)
  list_stimPaulis=[]
  for i in range(0,m): # for each row of the matrix
    x_list=[]
    z_list=[]
    for j in range(0,n):#for each element in the row
      x_list.append(int(matrix[i][j]))
      z_list.append(int(matrix[i][n+j]))
    xs = np.array(x_list, dtype=np.bool_)
    zs = np.array(z_list, dtype=np.bool_)
    pauli = stim.PauliString.from_numpy(xs=xs, zs=zs, sign=1) #use x and z part to make stimPauli
    list_stimPaulis.append(pauli)
  return list_stimPaulis

#Example
GF2 = galois.GF(2)
A = GF2([[1,1,0,0,0,1,1,0]])
print(MatrixToStimPaulis(A))

[stim.PauliString("+XYZ_")]


##Check Commutation of a group

We can check if a given list of Pauli operators all commute:

In [ ]:
#check whether a group of pauli operators commute. input is a list of stimPaulis
def IsStabilizerGroup(list_stimPaulis):
  count=0
  for i in range (0,len(list_stimPaulis)):
    for j in range(i,len(list_stimPaulis)):
      if list_stimPaulis[i].commutes(list_stimPaulis[j])==False:
        count+=1
        print('operator %d does not commute with operator %d, i.e. :'%(i,j))
        print(list_stimPaulis[i], ' and ', list_stimPaulis[j])
  if count==0:
    print('Everything Commutes!')

In [ ]:
IsStabilizerGroup(StringToStimPauli(['XIII','ZZZI','IXXX', 'IIIZ']))

operator 0 does not commute with operator 1, i.e. :
+X___  and  +ZZZ_
operator 2 does not commute with operator 3, i.e. :
+_XXX  and  +___Z


In [ ]:
IsStabilizerGroup(StringToStimPauli(['XZZXI','IXZZX','XIXZZ','ZXIXZ']))

Everything Commutes!


##Syndrome Extraction

We can check one pauli operators against a list of others, and record the commutation relations. This is precisely the goal of syndrome extraction for a quantum code

In [ ]:
#check if a stim pauli commutes or not with other stim paulis
def SyndromeExtract(stimPauli, list_stimPaulis):
  syndromes=[] # the syndrome is False if they commute
  for check in list_stimPaulis:
    syndromes.append(int(check.commutes(stimPauli)^True)) # we xor with True so that 'True' means 'anticommutes'
  return syndromes

#Example
SyndromeExtract(stim.PauliString('IXIZ'), StringToStimPauli(['XXXX', 'ZZZZ', 'XXZZ', 'ZZXX', 'IZIZ']))

[1, 1, 0, 0, 1]

##Naive Decoding Table

When we use a quantum stabilizer code, every Pauli operator is assigned some list of syndromes, based on the commutation relations with the generators. Let's write a function that returns a list of pairs of operators and syndromes.

In [ ]:
def SyndromeTableGenerator(list_stimPaulis,weight=0):
  n=list_stimPaulis[0].__len__()
  t=len(list_stimPaulis)
  pauli_string_iterator = stim.PauliString.iter_all(
  num_qubits=n, #length
  min_weight=0,
  max_weight=weight,
  allowed_paulis="XYZ",) #allowed operators
  table={}
  count=0 # count the number of Pauli operators
  for p in pauli_string_iterator:
    count+=1
    syndrome = SyndromeExtract(p,list_stimPaulis)
    table[tuple(syndrome)]=p
  print('There are %d available syndromes to assign'%(2**t))
  print('There are %d Pauli operators of weight up to %d'%(count,weight))
  return table

Here are the syndromes for the 5 qubit code:

In [ ]:
SyndromeTableGenerator(StringToStimPauli(['XZZXI','IXZZX','XIXZZ','ZXIXZ']), weight=1)

There are 16 available syndromes to assign
There are 16 Pauli operators of weight up to 1


{(0, 0, 0, 0): stim.PauliString("+_____"),
 (0, 0, 0, 1): stim.PauliString("+X____"),
 (1, 0, 1, 1): stim.PauliString("+Y____"),
 (1, 0, 1, 0): stim.PauliString("+Z____"),
 (1, 0, 0, 0): stim.PauliString("+_X___"),
 (1, 1, 0, 1): stim.PauliString("+_Y___"),
 (0, 1, 0, 1): stim.PauliString("+_Z___"),
 (1, 1, 0, 0): stim.PauliString("+__X__"),
 (1, 1, 1, 0): stim.PauliString("+__Y__"),
 (0, 0, 1, 0): stim.PauliString("+__Z__"),
 (0, 1, 1, 0): stim.PauliString("+___X_"),
 (1, 1, 1, 1): stim.PauliString("+___Y_"),
 (1, 0, 0, 1): stim.PauliString("+___Z_"),
 (0, 0, 1, 1): stim.PauliString("+____X"),
 (0, 1, 1, 1): stim.PauliString("+____Y"),
 (0, 1, 0, 0): stim.PauliString("+____Z")}

Here are the syndromes for the 7 qubit code:

In [ ]:
SyndromeTableGenerator(StringToStimPauli(['IIIXXXX','IXXIIXX','XIXIXIX', 'IIIZZZZ', 'IZZIIZZ', 'ZIZIZIZ']), weight=1)

There are 64 available syndromes to assign
There are 22 Pauli operators of weight up to 1


{(0, 0, 0, 0, 0, 0): stim.PauliString("+_______"),
 (0, 0, 0, 0, 0, 1): stim.PauliString("+X______"),
 (0, 0, 1, 0, 0, 1): stim.PauliString("+Y______"),
 (0, 0, 1, 0, 0, 0): stim.PauliString("+Z______"),
 (0, 0, 0, 0, 1, 0): stim.PauliString("+_X_____"),
 (0, 1, 0, 0, 1, 0): stim.PauliString("+_Y_____"),
 (0, 1, 0, 0, 0, 0): stim.PauliString("+_Z_____"),
 (0, 0, 0, 0, 1, 1): stim.PauliString("+__X____"),
 (0, 1, 1, 0, 1, 1): stim.PauliString("+__Y____"),
 (0, 1, 1, 0, 0, 0): stim.PauliString("+__Z____"),
 (0, 0, 0, 1, 0, 0): stim.PauliString("+___X___"),
 (1, 0, 0, 1, 0, 0): stim.PauliString("+___Y___"),
 (1, 0, 0, 0, 0, 0): stim.PauliString("+___Z___"),
 (0, 0, 0, 1, 0, 1): stim.PauliString("+____X__"),
 (1, 0, 1, 1, 0, 1): stim.PauliString("+____Y__"),
 (1, 0, 1, 0, 0, 0): stim.PauliString("+____Z__"),
 (0, 0, 0, 1, 1, 0): stim.PauliString("+_____X_"),
 (1, 1, 0, 1, 1, 0): stim.PauliString("+_____Y_"),
 (1, 1, 0, 0, 0, 0): stim.PauliString("+_____Z_"),
 (0, 0, 0, 1, 1, 1): stim.Pauli

Let's consider some formulas for the syndromes of an $[[n,k,d]]_2$ code

1. Number of syndromes is $2^{n-k}$.
2. Number of Pauli operators of weight $\omega$ is $\binom{n}{\omega}3^\omega$.
3. We can assign syndromes for operators up to weight $\lfloor \frac{d-1}{2} \rfloor$.

In [ ]:
from math import comb

In [ ]:
n,k,d = 13,8,3
num_syndromes = 2**(n-k)
num_correctable_errors = 0
for i in range(0,int(np.floor((d-1)/2))+1):
  num_correctable_errors += comb(n,i)*3**i

print(n-k)
print(num_syndromes)
print(num_correctable_errors)

5
32
40


## Syndrome-Correction Assigment

Suppose we have an $[[n,k,d]]_2$ code. There are $2^{n-k}$ syndromes, and $2^{2n}$ possible errors. Therefore, there are $2^{n+k}$ errors for each syndrome. When we want to apply a correction, we must choose which error we want to correct.

If we choose to correct error $E_s$ when we see syndrome $s$, then we will also apply a suitable correction to the $2^{n-k}$ errors which lie in the same stabilizer coset of $E_s$.

If an error $F$ occurs which is not in the same stabilizer coset as $E_s$, then this will **always** result in a logical error.

Usually, we choose $E_s$ to be the lowest weight operator of that syndrome. This works nicely for all syndromes which have an operator of weight $\le \lfloor \frac{d-1}{2}\rfloor$, as these operators all have distinct syndromes. For the leftover syndromes, we have to just pick some operator to correct.

I want to see all this work for the 5 qubit code; I want to compute all the cosets, and see what the weight distributions are like. What does minimum hamming weight assigment look like? What does the subfield metric look like?

In [ ]:
stabilizer=['XZZXI','IXZZX','XIXZZ','ZXIXZ']

In [ ]:
StimPaulisToMatrix(StringToStimPauli(stabilizer))

GF([[1, 0, 0, 1, 0, 0, 1, 1, 0, 0],
    [0, 1, 0, 0, 1, 0, 0, 1, 1, 0],
    [1, 0, 1, 0, 0, 0, 0, 0, 1, 1],
    [0, 1, 0, 1, 0, 1, 0, 0, 0, 1]], order=2)

In [ ]:
# #This function swaps the X and Z parts of the matrix
# def SWAPZX(matrix):


SyntaxError: incomplete input (2987756728.py, line 2)

##Compute Logical Z and Logical X operations

I already have magma code to do this, which can easily be used by flattening the galois matrix and copying it in.


In [ ]:
#Example
stabilizer=['XZZXI','IXZZX','XIXZZ',]

print(StringToStimPauli(stabilizer))
print(StimPaulisToMatrix(StringToStimPauli(stabilizer)))

A = StimPaulisToMatrix(StringToStimPauli(stabilizer)) # a 3x4 matrix
print(A.shape)
print(A.flatten().tolist())
# In magma, use: 'paritycheck:=Matrix(GF(2),3,4,[1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0]);'

[stim.PauliString("+XZZX_"), stim.PauliString("+_XZZX"), stim.PauliString("+X_XZZ")]
[[1 0 0 1 0 0 1 1 0 0]
 [0 1 0 0 1 0 0 1 1 0]
 [1 0 1 0 0 0 0 0 1 1]]
(3, 10)
[1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1]


In [ ]:
logicalZ = SpacedStringToPauli('0 0 0 0 0 1 1 1 1 1')
logicalX = SpacedStringToPauli('0 0 0 0 1 1 0 0 1 0')
print(logicalZ, logicalX)

+ZZZZZ +Z__ZX


Eventually I'll get around to doing it in python using the galois library. For now here is how I convert from magma back:

In Magma, use EltSeq(Matrix); to get a sequence e.g. [ 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1 ]

Then use Nrows() and Ncols() to get the dimension

In [ ]:
#Example
elements=[ 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1 ]
nrows=4
ncols=10
rows = [elements[i:i+ncols] for i in range(0, nrows*ncols, ncols)]
galois_matrix=GF2(rows)
galois_matrix

GF([[1, 0, 0, 0, 1, 1, 1, 0, 1, 1],
    [0, 1, 0, 0, 1, 0, 0, 1, 1, 0],
    [0, 0, 1, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 1, 1, 1, 0, 1, 1, 1]], order=2)

In [ ]:
print(MatrixToStimPaulis(galois_matrix))
IsStabilizerGroup(MatrixToStimPaulis(galois_matrix))

[stim.PauliString("+YZ_ZY"), stim.PauliString("+_XZZX"), stim.PauliString("+ZZX_X"), stim.PauliString("+Z_ZYY")]
Everything Commutes!


##Clifford Deformation

We can conjugate a set of stabilizer generators to get a new stabilizer code.


In [ ]:
# perform clifford deformation of a stabilizer code
def CliffordDeformedStabilizer(deformations, list_stimPaulis):
  # deformations should be a list of stim.CiruitInstruction() ; see the example below
  list_conjugates=[]
  for pauli in list_stimPaulis:
    for deformation in deformations:
      pauli=pauli.after(deformation)
    list_conjugates.append(pauli)
  return list_conjugates




In [ ]:
instruction1 = stim.CircuitInstruction('H', [0,1])
instruction2 = stim.CircuitInstruction('S', [3,4])
stabilizer = StringToStimPauli(['XXXXX', 'ZZZZZ', 'IIIII', 'YYYYY'])
CliffordDeformedStabilizer([instruction1,instruction2], stabilizer)

[stim.PauliString("+ZZXYY"),
 stim.PauliString("+XXZZZ"),
 stim.PauliString("+_____"),
 stim.PauliString("+YYYXX")]

In [ ]:
stim.PauliString('XX')*stim.PauliString('ZI')

stim.PauliString("-iYX")

##Stabilizer Equivalent Errors

We want a function that will return the minimum weight representation of a Pauli string up to stabilizer equivalence.

In [ ]:
#This function generates all elements of the stabilizer group
def GenerateStabilizerGroup(list_stim_paulis):
  full_stabilizer_group = []
  n = list_stim_paulis[0].__len__()
  t = len(list_stim_paulis)

  #prepare a blank pauli string
  stabilizer_string = ''
  for i in range(n):
    stabilizer_string += 'I'

  for i in range(2**t): #we sum over all binary strings of length t, to compute all possible stabilizer elements

    #we use i to create a binary string of length t
    digits = bin(i).split('b')[1] #this gives us the trailing digits of i; we need to pre append zeroes maybe
    if len(digits )<t:
      for k in range(0, t-len(digits )):
        digits  = '0' + digits

    coeffs = [int(d) for d in digits]
    #now we have a binary list of length t


    stabilizer_element = stim.PauliString(stabilizer_string) #we've created the identity pauli string
    #multiply in certain generator elements depending on which binary list we're in
    for j in range(0,t): # we check each coefficient to see if we should mutliply by that generator
      if coeffs[j] ==1:

        stabilizer_element= stabilizer_element*list_stim_paulis[j]
    full_stabilizer_group.append(stabilizer_element)

  return full_stabilizer_group


In [ ]:
#Example
stabilizer = StringToStimPauli(['XZZXI','IXZZX','XIXZZ','ZXIXZ'])
GenerateStabilizerGroup(stabilizer)

[stim.PauliString("+_____"),
 stim.PauliString("+ZX_XZ"),
 stim.PauliString("+X_XZZ"),
 stim.PauliString("+YXXY_"),
 stim.PauliString("+_XZZX"),
 stim.PauliString("+Z_ZYY"),
 stim.PauliString("+XXY_Y"),
 stim.PauliString("+Y_YXX"),
 stim.PauliString("+XZZX_"),
 stim.PauliString("+YYZ_Z"),
 stim.PauliString("+_ZYYZ"),
 stim.PauliString("+ZYYZ_"),
 stim.PauliString("+XY_YX"),
 stim.PauliString("+YZ_ZY"),
 stim.PauliString("+_YXXY"),
 stim.PauliString("+ZZX_X")]

In [ ]:
#This function returns a minimum weight equivalent error
def MinimumWeightEquivalentError(stimPauli,list_stim_paulis, full_list = False ):
  full_stabilizer_group = GenerateStabilizerGroup(list_stim_paulis)
  weights = []
  equiv_errors = []
  for i in range(len(full_stabilizer_group)):
    equiv_error = stimPauli*full_stabilizer_group[i]
    equiv_errors.append(equiv_error)
    weights.append(equiv_error.weight)

  min_index = int(np.argmin(weights))
  if full_list == True:
    return equiv_errors[min_index], equiv_errors
  else:
    return equiv_errors[min_index]


In [ ]:
#Example
stabilizer = StringToStimPauli(['XZZXI','IXZZX','XIXZZ','ZXIXZ'])
MinimumWeightEquivalentError(stim.PauliString('IZZIY'), stabilizer, full_list=True)

(stim.PauliString("+_ZZ_Y"),
 [stim.PauliString("+_ZZ_Y"),
  stim.PauliString("-ZYZXX"),
  stim.PauliString("-XZYZX"),
  stim.PauliString("-YYYYY"),
  stim.PauliString("+_Y_ZZ"),
  stim.PauliString("+ZZ_Y_"),
  stim.PauliString("+XYX__"),
  stim.PauliString("-YZXXZ"),
  stim.PauliString("+X__XY"),
  stim.PauliString("+YX__X"),
  stim.PauliString("+__XYX"),
  stim.PauliString("-ZXXZY"),
  stim.PauliString("-XXZYZ"),
  stim.PauliString("+Y_ZZ_"),
  stim.PauliString("+_XYX_"),
  stim.PauliString("+Z_Y_Z")])

In [ ]:
stim.PauliString('IZZIY')*stim.PauliString('ZXIXZ')*stim.PauliString('XIXZZ')

stim.PauliString("-YYYYY")